# AMPS II · Week 02 — HW2
## Plane stress and orthotropy, symmetry of a model, the axis of an axisymmetric model, and an LLM-generated pressure-vessel script

**Due before Week 3 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Plane stress, orthotropic compliance, rigid motions and the axis in $(r,z)$ (theory) | 20 |
| 2 | Orthotropic plate with a hole: plate size, fibre angle, quarter vs full model | 40 |
| 3 | Rotating shaft: the axis $r=0$, stresses on the axis, the axial force | 20 |
| 4 | Debugging an LLM-generated closed-end pressure-vessel script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run both cells once.

In [ ]:
%%writefile /tmp/fnx/hole.py
"""Plate with a circular hole (radius R) under tension T along x. Plane stress, isotropic or orthotropic.
quarter model: [0, L]^2 minus the disk, symmetry conditions on x = 0 and y = 0;  full model: [-L, L]^2 minus the disk."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LEFT, BOTTOM, RIGHT, TOP, HOLE = 1, 2, 3, 4, 5

def plate_mesh(R=0.1, L=1.0, h_hole=0.005, h_far=None, order=2, full=False):
    """Gmsh on rank 0 -> DOLFINx (API sheet §2). order=2: curved (isoparametric) triangles on the hole.
    Facet tags: 1 left, 2 bottom, 3 right, 4 top, 5 hole. Returns (msh, facet_tags)."""
    import gmsh
    from dolfinx.io import gmsh as gmshio
    h_far = h_far or 0.1 * L
    gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0)
    if comm.rank == 0:
        occ = gmsh.model.occ
        x0 = -L if full else 0.0
        s = occ.addRectangle(x0, x0, 0, L - x0, L - x0); d = occ.addDisk(0, 0, 0, R, R)
        occ.cut([(2, s)], [(2, d)])                          # removeObject/removeTool default True: nothing left over
        occ.synchronize()
        surf = gmsh.model.getEntities(2); gmsh.model.addPhysicalGroup(2, [t for _, t in surf], 1)
        tags = {k: [] for k in (LEFT, BOTTOM, RIGHT, TOP, HOLE)}
        for _, c in gmsh.model.getBoundary(surf, oriented=False):
            cx, cy, _ = occ.getCenterOfMass(1, c)
            k = (LEFT if np.isclose(cx, x0) else BOTTOM if np.isclose(cy, x0) else
                 RIGHT if np.isclose(cx, L) else TOP if np.isclose(cy, L) else HOLE)
            tags[k].append(c)
        for k, v in tags.items():
            gmsh.model.addPhysicalGroup(1, v, k)
        f = gmsh.model.mesh.field                            # element size: h_hole on the hole, growing to h_far
        dist = f.add("Distance"); f.setNumbers(dist, "CurvesList", tags[HOLE])
        th = f.add("Threshold"); f.setNumber(th, "InField", dist)
        f.setNumber(th, "SizeMin", h_hole); f.setNumber(th, "SizeMax", h_far)
        f.setNumber(th, "DistMin", 0.0); f.setNumber(th, "DistMax", 0.5 * L)
        f.setAsBackgroundMesh(th)
        for o in ("Mesh.MeshSizeExtendFromBoundary", "Mesh.MeshSizeFromPoints", "Mesh.MeshSizeFromCurvature"):
            gmsh.option.setNumber(o, 0)
        gmsh.model.mesh.generate(2)
        if order == 2:
            gmsh.model.mesh.setOrder(2)
    md = gmshio.model_to_mesh(gmsh.model, comm, 0, gdim=2)
    gmsh.finalize()
    return md.mesh, md.facet_tags

def area(msh):
    return comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=msh))), op=MPI.SUM)

eps = lambda u: ufl.sym(ufl.grad(u))

def isotropic(E=1.0, nu=0.3, plane="stress"):
    """sigma(u) in 2D: plane stress uses lambda* = 2 lambda mu / (lambda + 2 mu) = E nu / (1 - nu^2); plane strain uses lambda"""
    mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
    lam = 2 * lmbda * mu / (lmbda + 2 * mu) if plane == "stress" else lmbda
    return lambda u: 2 * mu * eps(u) + lam * ufl.tr(eps(u)) * ufl.Identity(2)

def orthotropic(E1, E2, nu12, G12, theta=0.0):
    """plane-stress orthotropic sigma(u); material axis 1 at angle theta (radians) from x.
    C = S^-1 in Voigt notation (engineering shear strain 2 eps_12); rotate the strain in, the stress out."""
    S = np.array([[1 / E1, -nu12 / E1, 0], [-nu12 / E1, 1 / E2, 0], [0, 0, 1 / G12]])
    C = ufl.as_matrix(np.linalg.inv(S).tolist())
    c, s = np.cos(theta), np.sin(theta)
    Q = ufl.as_matrix([[c, -s], [s, c]])                     # columns = material axes 1, 2 in (x, y)
    def sigma(u):
        e = Q.T * eps(u) * Q                                   # strain in material axes
        sv = C * ufl.as_vector([e[0, 0], e[1, 1], 2 * e[0, 1]])
        return Q * ufl.as_matrix([[sv[0], sv[2]], [sv[2], sv[1]]]) * Q.T
    return sigma

def lekhnitskii(E1, E2, nu12, G12):
    """Kt of an INFINITE orthotropic plate with a circular hole, tension along material axis 1 (isotropic: 3)"""
    return 1 + np.sqrt(2 * (np.sqrt(E1 / E2) - nu12) + E1 / G12)

def solve_plate(msh, ft, sigma, T=1.0, deg=2, full=False, L=1.0):
    """tension T along x.
    quarter: u_x = 0 on x = 0, u_y = 0 on y = 0 (symmetry), traction (T, 0) on x = L.
    full:    traction (+-T, 0) on x = +-L; three pinned DOFs at (-L, 0) and (L, 0) remove the rigid-body motions
             (the load is self-equilibrated, so the pins carry no force).
    Returns uh and info: Kt = max sigma_xx / T (sigma(u_h) interpolated into DG(deg-1)), the stress field s, DOFs, cells."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    t = fem.Constant(msh, np.array([T, 0.0]))
    a = ufl.inner(sigma(u), eps(v)) * ufl.dx
    if full:
        Lf = ufl.dot(t, v) * ds(RIGHT) - ufl.dot(t, v) * ds(LEFT)
        pL = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], -L) & np.isclose(x[1], 0.0))
        pR = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], L) & np.isclose(x[1], 0.0))
        bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(i), 0, pL), V.sub(i)) for i in (0, 1)]
        bcs.append(fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 0, pR), V.sub(1)))
    else:
        Lf = ufl.dot(t, v) * ds(RIGHT)
        bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(0), 1, ft.find(LEFT)), V.sub(0)),     # u_x = 0
               fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 1, ft.find(BOTTOM)), V.sub(1))]   # u_y = 0
    pr = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix="plate_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    W = fem.functionspace(msh, ("DG", max(deg - 1, 0), (2, 2)))
    s = fem.Function(W); s.interpolate(fem.Expression(sigma(uh), W.element.interpolation_points))
    sxx = s.x.array.reshape(-1, 4)[:, 0]
    Kt = comm.allreduce(sxx.max() if sxx.size else -np.inf, op=MPI.MAX) / T
    umax = comm.allreduce(np.abs(uh.x.array).max(), op=MPI.MAX)
    return uh, dict(Kt=Kt, s=s, umax=umax, reason=pr.solver.getConvergedReason(),
                    dofs=V.dofmap.index_map.size_global * 2, cells=msh.topology.index_map(2).size_global)

def argmax_angle(s):
    """polar angle (degrees) of the DG node with the largest sigma_xx"""
    X = s.function_space.tabulate_dof_coordinates(); sxx = s.x.array.reshape(-1, 4)[:, 0]
    k = int(np.argmax(sxx)) if sxx.size else 0
    best = comm.allreduce((float(sxx[k]) if sxx.size else -np.inf, float(np.degrees(np.arctan2(X[k, 1], X[k, 0])))), op=MPI.MAX)
    return best[1]

In [ ]:
%%writefile /tmp/fnx/axi.py
"""Axisymmetric linear elasticity in the (r, z) half-plane, u = (u_r, u_z), d/dtheta = 0."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, assemble_matrix, assemble_vector

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, TOP = 1, 2, 3, 4
E, nu = 1.0, 0.3
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))

def rz_mesh(ri, ro, h, nr, nz):
    """triangles on [ri, ro] x [0, h]. Facet tags: 1 r = ri (the axis if ri = 0), 2 r = ro, 3 z = 0, 4 z = h"""
    msh = mesh.create_rectangle(comm, [[ri, 0.0], [ro, h]], [nr, nz], cell_type=mesh.CellType.triangle)
    marks = [(INNER, lambda x: np.isclose(x[0], ri)), (OUTER, lambda x: np.isclose(x[0], ro)),
             (BOTTOM, lambda x: np.isclose(x[1], 0.0)), (TOP, lambda x: np.isclose(x[1], h))]
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, 1, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return msh, mesh.meshtags(msh, 1, fac, np.concatenate(val)[idx])

def eps_axi(u, r):
    """3D small strain in cylindrical components (rr, tt, zz): the hoop strain u_r / r is the new entry"""
    g = (u[0].dx(1) + u[1].dx(0)) / 2
    return ufl.as_matrix([[u[0].dx(0), 0, g], [0, u[0] / r, 0], [g, 0, u[1].dx(1)]])

def sigma_axi(u, r):
    e = eps_axi(u, r)
    return 2 * mu * e + lmbda * ufl.tr(e) * ufl.Identity(3)

def solve_axi(msh, ft, deg=2, tractions=(), body=None, fixed=()):
    """tractions: [(tag, (t_r, t_z))] (UFL or numbers); body: (f_r, f_z) or None; fixed: [(tag, component)] -> u_comp = 0.
    Returns uh and info with the forms a, L (needed for reactions) and r."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    r = ufl.SpatialCoordinate(msh)[0]
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = ufl.inner(sigma_axi(u, r), eps_axi(v, r)) * r * ufl.dx
    Lf = ufl.dot(fem.Constant(msh, np.zeros(2)), v) * r * ufl.dx
    for tag, t in tractions:
        Lf += ufl.dot(ufl.as_vector(t), v) * r * ds(tag)
    if body is not None:
        Lf += ufl.dot(ufl.as_vector(body), v) * r * ufl.dx
    bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 1, ft.find(tag)), V.sub(c)) for tag, c in fixed]
    pr = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix="axi_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    return uh, dict(a=a, L=Lf, r=r, ds=ds, msh=msh, ft=ft, reason=pr.solver.getConvergedReason(),
                    dofs=V.dofmap.index_map.size_global * 2)

def lame(r, ri, ro, pi, po=0.0, ends="open"):
    """thick cylinder (works with numbers and with UFL r): sigma_rr, sigma_tt, sigma_zz, u_r.
    ends: 'open' (sigma_zz = 0), 'closed' (end caps carry the pressure), 'plane_strain' (eps_zz = 0)"""
    A = (pi * ri**2 - po * ro**2) / (ro**2 - ri**2); B = (pi - po) * ri**2 * ro**2 / (ro**2 - ri**2)
    srr, stt = A - B / r**2, A + B / r**2
    szz = {"open": 0.0, "closed": A, "plane_strain": 2 * nu * A}[ends]
    return srr, stt, szz, r / E * (stt - nu * (srr + szz))

def solve_cylinder(n, deg, ri=1.0, ro=2.0, h=1.0, p=1.0, ends="open"):
    """internal pressure p, u_z = 0 on z = 0; 'plane_strain' also fixes u_z on z = h (open: top free).
    Returns uh, info with the r-weighted relative L2 error of u_r against Lame and the mean u_r on the inner wall."""
    msh, ft = rz_mesh(ri, ro, h, n, max(1, n // 2))
    fixed = [(BOTTOM, 1)] + ([(TOP, 1)] if ends == "plane_strain" else [])
    uh, info = solve_axi(msh, ft, deg, tractions=[(INNER, (p, 0.0))], fixed=fixed)   # internal pressure p
    r = info["r"]; ure = lame(r, ri, ro, p, 0.0, ends)[3]
    num = comm.allreduce(fem.assemble_scalar(fem.form((uh[0] - ure) ** 2 * r * ufl.dx)), op=MPI.SUM)
    den = comm.allreduce(fem.assemble_scalar(fem.form(ure ** 2 * r * ufl.dx)), op=MPI.SUM)
    ur_in = comm.allreduce(fem.assemble_scalar(fem.form(uh[0] * info["ds"](INNER))), op=MPI.SUM) / h
    return uh, dict(info, err=np.sqrt(num / den), ur_in=ur_in)

def residual_on(uh, info, tag):
    """r = K u - f with K, f assembled WITHOUT boundary conditions (API sheet §16c); returns (coordinates, r) of the
    DOFs on the facets 'tag' owned by this rank. These are the r-weighted equations: multiply by 2 pi for a force."""
    V = uh.function_space
    A0 = assemble_matrix(fem.form(info["a"])); A0.assemble()
    f0 = assemble_vector(fem.form(info["L"])); f0.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    res = A0.createVecLeft(); A0.mult(uh.x.petsc_vec, res); res.axpy(-1.0, f0)
    dofs = fem.locate_dofs_topological(V, 1, info["ft"].find(tag)); dofs = dofs[dofs < V.dofmap.index_map.size_local]
    return V.tabulate_dof_coordinates()[dofs], res.array.reshape(-1, 2)[dofs]

## Q1 — Theory (20 pts)

**(a) (7)** Plane stress: from $\sigma_{zz}=0$ derive $\varepsilon_{zz}$ in terms of $\varepsilon_{xx}+\varepsilon_{yy}$ and the in-plane law
$\boldsymbol\sigma=2\mu\boldsymbol\varepsilon+\lambda^*(\operatorname{tr}\boldsymbol\varepsilon)\mathbf I$ with $\lambda^*=2\lambda\mu/(\lambda+2\mu)$.
What happens to $\lambda$ and to $\lambda^*$ as $\nu\to\tfrac12$? What does this mean for volumetric locking (Week 1, Part A) in plane stress and in plane strain?

**(b) (6)** Orthotropic plane stress: the compliance $\mathbf S$ (Voigt, engineering shear) has $S_{12}=-\nu_{12}/E_1$ and $S_{21}=-\nu_{21}/E_2$. Why must
$\mathbf S$ be symmetric, and what relation follows? Show that $\mathbf S$ is positive definite iff $E_1,E_2,G_{12}>0$ and $\nu_{12}^2<E_1/E_2$.

**(c) (7)** Axisymmetry. (i) Which of the six 3D rigid-body motions are axisymmetric with $u_\theta=0$, i.e. survive in the $(r,z)$ model? Show that
$u_r=c$ (constant) is **not** a rigid motion. (ii) For a solid body ($r=0$ inside $\Omega$), show that a finite strain energy needs $u_r\to0$ on the axis.
Predict: is the Dirichlet condition $u_r=0$ on the axis necessary in the FE model? Q3(a) tests your prediction.

> *your answer*

## Q2 — Orthotropic plate with a hole (40 pts)

Material of lab §4 ($E_1=200\,000$, $E_2=20\,000$, $\nu_{12}=0.3$, $G_{12}=10\,000$), $R=0.1$, tension along $x$, P2 on curved triangles.

**(a) (12)** Plate size. For $\theta=0°$ and $90°$, compute $K_t$ on the quarter plate for $L/R=10,20,40$ (`plate_mesh(L=..., h_hole=0.005)`, `solve_plate(..., L=...)`) and
compare with Lekhnitskii. How does the convergence to the infinite-plate value compare with the isotropic one (lecture: 3.085, 3.021, 3.005)?

**(b) (16)** Fibre angle $\theta=15°$. Compute $K_t$ with the **quarter** model and with the **full** plate (`full=True`) on three meshes:
(`h_hole`, `h_far`) = (0.005, 0.1), (0.0025, 0.05), (0.00125, 0.025). Do the same for $\theta=0°$. Which results converge? Explain what is wrong with one of the
two models, and why it is not wrong for $\theta=0°$.

**(c) (12)** With the model you trust, compute $K_t(\theta)$ for $\theta=0,15,30,45,60,75,90°$ (`h_hole=0.00125, h_far=0.025`) and the polar angle of the point of
maximum $\sigma_{xx}$ (`argmax_angle(info["s"])`). Plot $K_t(\theta)$ with the two Lekhnitskii values. Comment on the trend and on the location.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — plate size study for theta = 0 and 90 degrees

> *your explanation (Q2a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) — quarter vs full plate, theta = 15 and 0 degrees, three meshes

> *your explanation (Q2b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) — Kt(theta) and the location of the maximum (full plate); save the table to /tmp/fnx/w02_q2c.json for the plot below

In [ ]:
# TODO: Q2(c) — plot Kt(theta) with the two Lekhnitskii values

> *your explanation (Q2c)*

## Q3 — Rotating shaft: the axis (20 pts)

A solid cylinder $0\le r\le b=1$, $0\le z\le0.5$ rotates about its axis; the centrifugal body force is $f_r=\rho\omega^2r$ with $\rho\omega^2=1$, $E=1$, $\nu=0.3$.
$u_z=0$ on $z=0$ **and** $z=0.5$ (plane strain). The exact solution is
$$u_r=\frac{\rho\omega^2\,r}{8(\lambda+2\mu)}\Big(\frac{2\lambda+3\mu}{\lambda+\mu}\,b^2-r^2\Big),\qquad u_z=0 .$$
Use `rz_mesh(0.0, 1.0, 0.5, n, n//2)` (the tag `INNER` is then the axis) and `solve_axi(msh, ft, deg, body=(r, 0.0), fixed=[...])` with `r = ufl.SpatialCoordinate(msh)[0]`
— note that `solve_axi` adds the weight $r$ itself.

**(a) (8)** For P1 and P2, $n=4,8,16,32,64$: the relative $L^2_r$ error of $u_r$ **with** and **without** the condition $u_r=0$ on the axis, and $\max|u_r|$ on the axis. Rates?
Was your prediction in Q1(c) right?

**(b) (6)** Stresses on the axis: $\sigma_{rr}(0)=\sigma_{\theta\theta}(0)$. Interpolate $\sigma_{\theta\theta}$ of the P2 solution ($n=16$, with the axis condition) into `DG 1` and into `DG 0`;
count the non-finite values and compare the largest finite value with the exact $\sigma_{\theta\theta}(0)$ (derive it). Explain what you see. How would you report the stress on the axis?

**(c) (6)** Plane strain needs an axial force on the top face $z=0.5$. Compute it from the consistent reaction (`residual_on(uh, info, TOP)`) and compare with the
exact value $F_z=2\pi\int_0^b\sigma_{zz}\,r\,dr$. *(Hint: for this solution $\sigma_{zz}=\lambda(u_r'+u_r/r)$ and $\int_0^b(u_r'+u_r/r)\,r\,dr=b\,u_r(b)$.)*

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a)

> *your explanation (Q3a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q3(b)

> *your explanation (Q3b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q3(c)

> *your explanation (Q3c)*

## Q4 — Debugging an LLM-generated pressure-vessel script (20 pts)

An LLM was asked for "an axisymmetric model of a closed-end thick-walled steel cylinder, $r_i=0.5$ m, $r_o=1$ m, internal pressure 20 MPa, P2,
printing the radial displacement and hoop stress at the inner wall and the mean axial stress". The cell below is what it produced. It contains
**four** defects.

**(a) (8)** Run it, then find and describe all four defects (what is wrong, how you found it).
**(b) (8)** Write the corrected script in the cell after it and run it.
**(c) (4)** Verify the corrected script against the closed-end Lamé solution ($u_r(r_i)$, $\sigma_{\theta\theta}(r_i)$, $\sigma_{zz}$) and with a force balance:
the axial reaction on $z=0$ (consistent, with $2\pi$) must equal the end-cap force $p\pi r_i^2$.

In [ ]:
%%fenicsx -np 2
# LLM-generated script (HW2 Q4) — it runs, but it contains four defects
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
ri, ro, h = 0.5, 1.0, 1.0                 # m
E, nu, p = 200e3, 0.3, 20.0               # MPa
mu = E / (2 * (1 + nu))
lmbda = E * nu / (1 - nu**2)              # Lame parameter

msh = mesh.create_rectangle(comm, [[ri, 0.0], [ro, h]], [16, 16], cell_type=mesh.CellType.triangle)
V = fem.functionspace(msh, ("Lagrange", 2, (2,)))
x = ufl.SpatialCoordinate(msh); r = x[0]

def eps(u):   # strain in (rr, tt, zz) components
    return ufl.as_matrix([[u[0].dx(0), 0, (u[0].dx(1) + u[1].dx(0)) / 2],
                          [0, u[0] / r, 0],
                          [(u[0].dx(1) + u[1].dx(0)) / 2, 0, u[1].dx(1)]])
def sigma(u):
    return 2 * mu * eps(u) + lmbda * ufl.tr(eps(u)) * ufl.Identity(3)

inner_f = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], ri))
top_f = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], h))
bot_f = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], 0.0))
fac = np.r_[inner_f, top_f]; tag = np.r_[np.full(len(inner_f), 1), np.full(len(top_f), 2)].astype(np.int32)
o = np.argsort(fac); ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, fac[o], tag[o]))

n = ufl.FacetNormal(msh)
t_in = p * n                                # pressure on the inner wall
t_cap = p * ri**2 / ro**2                   # axial stress from the end cap (closed vessel)
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
a = ufl.inner(sigma(u), eps(v)) * r * ufl.dx
L = ufl.dot(t_in, v) * ds(1) + t_cap * v[1] * r * ds(2)
bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 1, bot_f), V.sub(1))
uh = LinearProblem(a, L, bcs=[bc], petsc_options_prefix="q4_",
                   petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()

ur_in = comm.allreduce(fem.assemble_scalar(fem.form(uh[0] * ds(1))), op=MPI.SUM) / h
stt_in = comm.allreduce(fem.assemble_scalar(fem.form(sigma(uh)[1, 1] * ds(1))), op=MPI.SUM) / h
szz = comm.allreduce(fem.assemble_scalar(fem.form(sigma(uh)[2, 2] * r * ds(2))), op=MPI.SUM) * 2 / (ro**2 - ri**2)
if comm.rank == 0:
    print(f"radial displacement at r_i : {abs(ur_in):.4e} m")
    print(f"hoop stress at r_i         : {abs(stt_in):.2f} MPa")
    print(f"mean axial stress          : {abs(szz):.2f} MPa")
    print(f"thin-wall check p*r_m/t    : {p * (ri + ro) / 2 / (ro - ri):.2f} MPa (thick wall: expect ~10-20 % higher)")

> *your list of defects (Q4a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) — the corrected script

In [ ]:
%%fenicsx -np 2
# TODO: Q4(c) — verification: closed-end Lame and the axial force balance

> *your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …